In [1]:
!pip install transformers --quiet


In [28]:
import pandas as pd
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, Conv1D, GlobalMaxPooling1D,
                                     Dense, LSTM, SimpleRNN, Bidirectional, Dropout)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
file_path = '/content/drive/MyDrive/efsa_sentiment_classification.csv'

news_data = pd.read_csv(file_path)
news_data.head()
news_data.shape

(1173, 8)

In [15]:
df = pd.read_csv(file_path)

# keep only 'NEG', 'POS'
df = df[df['Sentiment'].isin(['NEG', 'POS'])].copy()


texts = df['Sentence'].astype(str).tolist()
labels = df['Sentiment'].map({'NEG': 0, 'POS': 1}).values


In [16]:
print(df['Sentiment'].value_counts(dropna=False))


Sentiment
POS    760
NEG    399
Name: count, dtype: int64


In [17]:
# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)


In [18]:
max_words = 10000
max_len   = 100

tokenizer = Tokenizer(num_words=max_words, oov_token='<UNK>')
tokenizer.fit_on_texts(X_train)

train_sequences = tokenizer.texts_to_sequences(X_train)
val_sequences = tokenizer.texts_to_sequences(X_val)

X_train_pad = pad_sequences(train_sequences, maxlen=max_len, padding='post', truncating='post')
X_val_pad = pad_sequences(val_sequences,   maxlen=max_len, padding='post', truncating='post')


In [19]:
# Convert integer labels to one-hot encoding for categorical crossentropy
y_train_cat = to_categorical(y_train, num_classes=2)
y_val_cat   = to_categorical(y_val,   num_classes=2)

In [20]:
# 4. Define three baseline models: CNN, Simple RNN, and BiLSTM
def build_cnn_model():
    # CNN-based text classification model
    model = Sequential([
        Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
        Conv1D(filters=128, kernel_size=5, activation='relu'),
        GlobalMaxPooling1D(),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dense(2, activation='softmax')
    ])
    return model

In [21]:
def build_rnn_model():
    # Simple RNN-based text classification model
    model = Sequential([
        Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
        SimpleRNN(64),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dense(2, activation='softmax')
    ])
    return model

In [22]:
def build_lstm_model():
    # Bidirectional LSTM-based text classification model
    model = Sequential([
        Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
        Bidirectional(LSTM(64)),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dense(2, activation='softmax')
    ])
    return model

In [31]:
# Train and evaluate each model
for name, build_fn in [
    ('CNN',   build_cnn_model),
    ('RNN',   build_rnn_model),
    ('LSTM',  build_lstm_model)
]:
    print(f'\nTraining {name} model')
    model = build_fn()
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=2,
    restore_best_weights=True)


    history = model.fit(
        X_train_pad, y_train_cat,
        validation_data=(X_val_pad, y_val_cat),
        epochs=5,
        batch_size=32,
        callbacks=[early_stopping]
    )

    loss, accuracy = model.evaluate(X_val_pad, y_val_cat, verbose=0)
    print(f'{name} validation accuracy: {accuracy:.4f}')


Training CNN model
Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


29/29 ━━━━━━━━━━━━━━━━━━━━ 5s 72ms/step - accuracy: 0.5625 - loss: 0.6754 - val_accuracy: 0.6552 - val_loss: 0.6457
Epoch 2/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.6619 - loss: 0.6099 - val_accuracy: 0.6767 - val_loss: 0.6081
Epoch 3/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7070 - loss: 0.5007 - val_accuracy: 0.7371 - val_loss: 0.5115
Epoch 4/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9719 - loss: 0.2285 - val_accuracy: 0.7716 - val_loss: 0.4961
Epoch 5/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9904 - loss: 0.0555 - val_accuracy: 0.7888 - val_loss: 0.5831
CNN validation accuracy: 0.7888

Training RNN model
Epoch 1/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 6s 88ms/step - accuracy: 0.6464 - loss: 0.6557 - val_accuracy: 0.6552 - val_loss: 0.6658
Epoch 2/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.6459 - loss: 0.6540 - val_accuracy: 0.6250 - val_loss: 0.6518
Epoch 3/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6369 - loss: 0.6